## MNIST met Linear Regression

Bij K-means onthielden we gemiddelde plaatjes.
Nu doen we iets anders:

- We leren per cijfer een formule die zegt hoe waarschijnlijk het is dat een plaatje dat cijfer is.

Dat klinkt spannend, maar het komt neer op:

- pixels krijgen een gewicht

- sommige pixels tellen zwaarder mee dan andere

### Laden MNIST

Je mag je code uit de Kmeans hergebruiken waar van toepassing.

Schrijf een functie die:

- MNIST laadt

- pixelwaarden schaalt naar 0–1

- labels omzet naar gehele getallen

**Vragen:**

Waarom schalen we pixelwaarden?

Hoeveel features heeft één afbeelding?

In [24]:
# imports
import numpy as np;
from tensorflow.keras.datasets import mnist;

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [25]:
def normalize_and_vectorize_image(image: np.array):
    """
    Turn a image of variable dimesions (must be square) into a vector and normalize values between 0 and 1
    """
    flat = image.flatten()

    maximum = np.max(flat)
    normalised: np.array = flat / maximum
    return normalised

In [26]:
train_images_vectorized = np.array([normalize_and_vectorize_image(img) for img in train_images])
test_images_vectorized = np.array([normalize_and_vectorize_image(img) for img in test_images])
print("finished")

finished


In [27]:
print(f"Aantal features per afbeelding: {len(train_images_vectorized[0])}")

Aantal features per afbeelding: 784


**Antwoorden op de bovenstaande vragen:**

Waarom schalen we pixelwaarden?

_Door de pixelwaarden te schalen wordt het vergelijken van de pixels tussen verschillende afbeeldingen eerlijker._

Hoeveel features heeft één afbeelding?

_Elke afbeelding bestaat uit een 2d array. Hierboven heb ik een stukje code geschreven die de aantal pixels telt. dit zijn ook de aantal features_ 


&nbsp;



### Labels geschikt maken voor regressie

Linear regression werkt met getallen, niet met categorieën.

Waarom is dat?

_Lineare regressie leert een vergelijking door berekeningen te maken. Met categorieen (zoals "rood", "blauw", of "groen") kun je geen berekeningen maken. er zijn dus numerieke waardes nodig._

We doen daarom one-vs-rest:

- eg voor cijfer 3:

- alle 3’en → 1

- alle andere cijfers → 0

Schrijf een functie die:

- één gekozen cijfer omzet naar binaire labels, dus 
    - 1 voor het gekozen cijfer
    - 0 voor alle andere cijfers

Vragen:

Waarom kunnen we niet direct 0–9 voorspellen?

_In dit geval zijn de cijfers 0-9 categorieen. Lineare regressie geeft een score / voorspelde waarde terug en op basis van die score / waarde kan je door one vs rest de vraag stellen "is dit een 3"_


Wat betekent een voorspelling van 0.8?

_Hoe dichter de waarde bij de 1 zit, hoe sterker de voorspelling is. als je een waarde van 0.8 krijgt denkt het model sterk dat het X getal is._

In [28]:
def make_binary_labels(labels: np.array, positive_label: int):
    """
    Turn a list of labels into binary labels. The positive label is the one specified, the negative label is all other labels.
    """
    binary_labels = np.where(labels == positive_label, 1, 0)
    return binary_labels

### Lineair model trainen

We leren een gewicht per pixel.

Intuïtief:

- heldere pixels op logische plekken krijgen hoge gewichten

- achtergrond krijgt lage gewichten


Je mag de volgende functie gebruiken:

```python
def train_linear_model(X_train, y_binary):
    model = LinearRegression()
    model.fit(X_train, y_binary)
    return model
```
Kort samengevat: Linearie Regressie berekent exact de optimale waarden voor **intercept** & **coefficients**.

Vragen:

- Wat is X_train hier?  
_in dit geval een mnist image als 1D array. (dus 1 array van 784px in plaats van een matrix van 28x28px)_

- En y_binary?  
_ y_binary is de doelouput van de labels geencode in one vs rest. (1 representeert het gekozen cijfer, 0 alle andere cijfers)_

- Wat stelt één gewicht voor?  
_ één gewicht is kortgezegd het belang van een pixel._  
_hoge positieve waarde → pixel is belangrijk voor “ja”_  
_negatieve waarde → pixel wijst juist op “nee”_  
_bijna 0 → pixel is niet belangrijk_
  
_waar het cijfer vaak zwart is → hoge gewichten_
_achtergrond → lage of nul gewichten_

- Hoeveel gewichten heeft het model? Tip: kijk naar **model.coef_**  
_elke afbeelding is 28x28 = 784 pixels, dus 784 gewichten_

- Wat is model.intercept_ denk je?  
_Dit is de basiswaarde van het model: De voorspelling als alle pixels 0 zouden zijn.

In [37]:
from sklearn.linear_model import LinearRegression

def train_linear_model(X_train, y_binary):
    model = LinearRegression()
    model.fit(X_train, y_binary)
    # print(f"coef {model.coef_.shape}")
    return model

### Model trainen voor 1 cijfer

Laten we eerst het model trainen voor een cijfer. 
Stop letterlijk alle 784 (genormaliseerde) pixelwaardes als features in het model
Schrijf code die dat doet

In [38]:
# loaded mnist is een tuple die komt van

def single_experiment(digit, loaded_mnist):
    (train_images, train_labels), _ = loaded_mnist

    # 1. Maak binary labels
    binary_labels = make_binary_labels(train_labels, digit)

    # 2. Vectorize + normalize
    X_train = np.array([
        normalize_and_vectorize_image(img)
        for img in train_images
    ])

    # 3. Train model
    model = train_linear_model(X_train, binary_labels)

    # print(model.coef_.shape)
    # print(model.intercept_)

    return model

single_experiment(1, ((train_images, train_labels), (test_images, test_labels)))


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


### Een cijfer die we getraind hebben voorspellen

Voorspel nu of een test cijfer inderdaad is wat je verwacht.

Maak een functie predict_num met de volgende argumenten:
- model
- image
- echte image waarde

Leg uit waarom we deze 3 nodig hebben?

Je kunt voor het voorspellen de volgende functie gebruiken:
```python
    score = model.predict([image])[0]
```

Waarom is die [0] er? zoek het uit

Score is afhankelijk van het model. In dit geval kun je het lezen als:

- Score > 0 → model denkt: dit is waarschijnlijk een 5

- Score < 0 → model denkt: dit is geen 5

- |score| groot → model is zekerder

In [39]:
def predict_num(model, image, digit):            
    image_vec = normalize_and_vectorize_image(image)

    score = model.predict([image_vec])[0]

    prediction = 1 if score > 0 else 0

    # Echte waarde (ground truth)
    actual = 1 if digit == 1 else 0

    print(f"Score: {score}")
    print(f"Prediction: {prediction}")
    print(f"Actual: {actual}")

    return prediction, actual, score

In [44]:
randint = np.random.randint(0, len(test_images))

model = single_experiment(1, ((train_images, train_labels), (test_images, test_labels)))

predict_num(model, test_images[randint], test_labels[randint])

Score: 0.13651717670348118
Prediction: 1
Actual: 0


(1, 0, np.float64(0.13651717670348118))

### Experimenteren op een echt getal

Train het nu op bijvoorbeeld getal 5.

Maak nu code waarbij we een random getal uit de database halen en deze classificeren. Je mag de volgende code gebruiken om het getal te tonen:

```python
plt.imshow(random_image.reshape(28, 28), cmap="gray")
plt.title(f"Echte label: {true_label}")
plt.axis("off")
plt.show()
```


In [33]:
#Train het model op getal 5
digit = 5
model = single_experiment(digit, (X, y))

NameError: name 'X' is not defined

### Modellen trainen voor alle cijfers

We trainen 10 aparte modellen:

- één voor 0

- één voor 1

- etc...

Schrijf een functie die:

- voor elk cijfer een model traint

- alle modellen opslaat in een datastructuur

### Voorspellen

Een nieuw plaatje:

- gaat door alle 10 modellen

- elk model geeft een score

- hoogste score wint

Schrijf een functie **predict_case** die:

- voor elk testplaatje voorspelt welk cijfer het is

- gebruik **model.predict([image])**

Vragen:

Wat betekent een hoge score?

Waarom kiezen we de hoogste score?

### Gebruik nu je eigen features

Gebruik een functie uit de vorige opdrachten om je eigen features te gebruiken voor linear regression.


def extract_features(img):
    features = []
    features.append(img.mean())                      # gemiddelde intensiteit
    features.append((img > 0.5).sum())               # aantal "actieve" pixels
    ...
    ....
    return features


## Rapporteer: 




De gemaakte opdrachten en de discussie/feedbackmoeten worden uitgewerkt en op CodeGrade geplaatst onder **P6 Linear Regression**